# ML-06 — Signal Audit: Do the Flags Hold?

This is a public-safe exploratory audit of the anonymized starter slice. It describes associations, not causes or guaranteed editorial outcomes.

## 1. Distributions

Search-demand fields are expected to be heavy-tailed, so median summaries and log-aware downstream features are safer than treating raw counts as normally distributed.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

cols = [c for c in ['impressions_90d','clicks_90d','sessions_90d','avg_position'] if c in df]
print(df[cols].describe(percentiles=[.5,.9]).loc[['count','50%','90%']].round(2).to_string())

       impressions_90d  clicks_90d  sessions_90d  avg_position
count          30000.0     30000.0       30000.0       30000.0
50%              731.0         1.0           7.0          10.8
90%            12136.4        32.0          88.0          36.8


## 2. Data-quality checks

`avg_position == 0` means no ranking data, not first position. Rate columns are stored as percentage points, and missingness should be preserved as information rather than blindly replaced with zero.

In [2]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

print('avg_position == 0 (no-data rows)=', int((df.avg_position == 0).sum()))
print('missing word_count=', int(df.word_count.isna().sum()))
print('total missing cells=', int(df.isna().sum().sum()))

avg_position == 0 (no-data rows)= 1205
missing word_count= 7699
total missing cells= 73868


## 3. Directional signal check

Compare the decline-proxy rate across broad, non-identifying content types. This is a diagnostic stratification only; it does not prove that content type causes decline and does not replace grouped validation.

In [3]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

summary = df.groupby('content_type')['is_declining_label'].agg(['count','mean']).sort_values('count', ascending=False)
print(summary.rename(columns={'mean':'decline_proxy_rate'}).round(3).to_string())

                    count  decline_proxy_rate
content_type                                 
keyword article     27207               0.561
feedly article       2096               0.287
comparison article    697               0.572


## 4. Verdict and next action

The slice has usable observed signals but also strong skew, no-data positions, and structured missingness. The right next step is a leakage-aware baseline and grouped-client validation—not automatic content changes. Pages that rank highly should receive a human check of intent, SERP context, technical health, facts, and brand fit.

## Self-check

- [x] Distributions, data quality, and a grouped directional check are computed.
- [x] No client names, URLs, or raw queries are displayed.
- [x] Findings are associative and decision-support only.

In [4]:
print('Signal-audit verdict: usable with skew, missingness, and leakage controls.')

Signal-audit verdict: usable with skew, missingness, and leakage controls.
